# NCTB MCQ — Preprocessing Pipeline
Loads `nayemislamzr/NCTB_MCQ` from HuggingFace, parses questions/choices/answers, validates, and exports train/val/test splits.

In [1]:
import sys
import os

# Add repo root to path so src.dataloader is importable
REPO_ROOT = os.path.abspath(os.path.join(os.getcwd(), '..'))
if REPO_ROOT not in sys.path:
    sys.path.insert(0, REPO_ROOT)

DATA_OUT = os.path.join(REPO_ROOT, 'data', 'processed')
print('Repo root:', REPO_ROOT)
print('Output dir:', DATA_OUT)

Repo root: /Users/kaizu/All/Projects/BengaliLLM
Output dir: /Users/kaizu/All/Projects/BengaliLLM/data/processed


## Cell 1 — Load raw dataset and inspect

In [3]:
from src.dataloader import load_raw_as_dataframe

# Loads train split from HuggingFace (8,480 rows)
# Columns: source, problem, solution, messages
df_raw = load_raw_as_dataframe()
print('Shape:', df_raw.shape)
print('Columns:', df_raw.columns.tolist())
df_raw.head(2)

/opt/miniconda3/envs/env_AI/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Generating train split: 100%|██████████| 8478/8478 [00:00<00:00, 18910.22 examples/s]

Shape: (8478, 4)
Columns: ['source', 'problem', 'solution', 'messages']


,source,problem,solution,messages
0,NCTB,নিচের কোনটি 3 × 2 ক্রমের ম্যাট্রিক্স?\nক: \beg...,একটি ম্যাট্রিক্সের ক্রম (Order) হলো সারি সংখ্য...,"[{""content"": ""নিচের কোনটি 3 × 2 ক্রমের ম্যাট্র..."
1,NCTB,A = $[a_{ij}]_{m \times n}$ বর্গ ম্যাট্রিক্স হ...,একটি ম্যাট্রিক্সকে বর্গ ম্যাট্রিক্স বলা হয় যদ...,"[{""content"": ""A = $[a_{ij}]_{m \\times n}$ বর্..."


## Cell 2 — Inspect a single raw row

In [4]:
# Check raw problem and solution text for one row to understand format
idx = 0
print('--- PROBLEM ---')
print(df_raw.loc[idx, 'problem'])
print('\n--- SOLUTION ---')
print(df_raw.loc[idx, 'solution'])

--- PROBLEM ---
নিচের কোনটি 3 × 2 ক্রমের ম্যাট্রিক্স?
ক: \begin{pmatrix} x & y & z \\ 1 & 2 & 3 \end{pmatrix}
খ: \begin{pmatrix} 2 & 4 \\ 0 & 5 \\ 6 & 0 \end{pmatrix}
গ: \begin{pmatrix} 1 & 2 & 3 \end{pmatrix}
ঘ: \begin{pmatrix} a & b & c \\ d & e & f \\ g & h & f \end{pmatrix}

--- SOLUTION ---
একটি ম্যাট্রিক্সের ক্রম (Order) হলো সারি সংখ্যা × কলাম সংখ্যা। 3 × 2 ক্রমের ম্যাট্রিক্সের 3টি সারি এবং 2টি কলাম থাকবে।
বিকল্পসমূহ পরীক্ষা করা যাক:
ক. \begin{pmatrix} x & y & z \\ 1 & 2 & 3 \end{pmatrix} - এই ম্যাট্রিক্সের 2টি সারি ও 3টি কলাম আছে, তাই এটি 2 × 3 ক্রমের।
খ. \begin{pmatrix} 2 & 4 \\ 0 & 5 \\ 6 & 0 \end{pmatrix} - এই ম্যাট্রিক্সের 3টি সারি ও 2টি কলাম আছে, তাই এটি 3 × 2 ক্রমের।
গ. \begin{pmatrix} 1 & 2 & 3 \end{pmatrix} - এই ম্যাট্রিক্সের 1টি সারি ও 3টি কলাম আছে, তাই এটি 1 × 3 ক্রমের।
ঘ. \begin{pmatrix} a & b & c \\ d & e & f \\ g & h & f \end{pmatrix} - এই ম্যাট্রিক্সের 3টি সারি ও 3টি কলাম আছে, তাই এটি 3 × 3 ক্রমের।
সুতরাং, সঠিক উত্তর হলো 'খ'।

\boxed{খ: \begin{pmatrix} 2 & 4 \\ 0 &

## Cell 3 — Test parsers on a few examples

In [5]:
from src.dataloader.parser import parse_question, parse_choices, extract_answer

# Verify parser outputs on first 3 rows before running full pipeline
for i in range(3):
    row = df_raw.iloc[i]
    q  = parse_question(row['problem'])
    ch = parse_choices(row['problem'])
    ans = extract_answer(row['solution'])
    print(f'[{i}] question : {q[:60] if q else None}...')
    print(f'[{i}] choices  : {ch}')
    print(f'[{i}] answer   : {ans}')
    print()

[0] question : নিচের কোনটি 3 × 2 ক্রমের ম্যাট্রিক্স?...
[0] choices  : {'A': '\\begin{pmatrix} x & y & z \\\\ 1 & 2 & 3 \\end{pmatrix}', 'B': '\\begin{pmatrix} 2 & 4 \\\\ 0 & 5 \\\\ 6 & 0 \\end{pmatrix}', 'C': '\\begin{pmatrix} 1 & 2 & 3 \\end{pmatrix}', 'D': '\\begin{pmatrix} a & b & c \\\\ d & e & f \\\\ g & h & f \\end{pmatrix}'}
[0] answer   : B

[1] question : A = $[a_{ij}]_{m \times n}$ বর্গ ম্যাট্রিক্স হলে m ও n এর মধ...
[1] choices  : {'A': 'm>n', 'B': 'm<n', 'C': 'm=n', 'D': 'm = -n'}
[1] answer   : C

[2] question : 2 × m মাত্রার ম্যাট্রিক্সের কলাম সংখ্যা কত?...
[2] choices  : {'A': 'বর্গ', 'B': 'কর্ণ', 'C': 'স্কেলার', 'D': 'আয়তাকার'}
[2] answer   : D



## Cell 4 — Run full preprocessing pipeline

In [6]:
from src.dataloader import preprocess

# Applies parse_question, parse_choices, extract_answer to all rows
# Adds: id, domain='education', source='NCTB_MCQ'
df_processed = preprocess(df_raw)
print('Shape after preprocessing:', df_processed.shape)
df_processed[['id', 'domain', 'question', 'choices', 'answer']].head(3)

Shape after preprocessing: (8478, 9)


,id,domain,question,choices,answer
0,edu_0000,education,নিচের কোনটি 3 × 2 ক্রমের ম্যাট্রিক্স?,{'A': '\begin{pmatrix} x & y & z \\ 1 & 2 & 3 ...,B
1,edu_0001,education,A = $[a_{ij}]_{m \times n}$ বর্গ ম্যাট্রিক্স হ...,"{'A': 'm>n', 'B': 'm<n', 'C': 'm=n', 'D': 'm =...",C
2,edu_0002,education,2 × m মাত্রার ম্যাট্রিক্সের কলাম সংখ্যা কত?,"{'A': 'বর্গ', 'B': 'কর্ণ', 'C': 'স্কেলার', 'D'...",D


## Cell 5 — Validate and inspect drop report

In [7]:
from src.dataloader import validate

# Drops rows with missing question/choices/answer or low Bengali ratio
# report contains per-reason drop counts
df_valid, report = validate(df_processed)
print('Validation report:', report)
print('Answer distribution:\n', df_valid['answer'].value_counts())

Validation report: {'missing_question': 0, 'missing_choices': 114, 'missing_answer': 0, 'low_bengali': np.int64(402), 'total_dropped': 516, 'remaining': 7962}
Answer distribution:
 answer
A    2580
B    2068
C    1812
D    1502
Name: count, dtype: int64


## Cell 6 — Split (70/15/15) and export to data/processed/

In [8]:
from src.dataloader import split_and_export

# Saves train.jsonl, val.jsonl, test.jsonl to DATA_OUT
# Each line is a JSON object with: id, domain, question, choices, answer, source
result = split_and_export(df_valid, out_dir=DATA_OUT)

print(f"Train : {result['train']} rows -> {result['paths']['train']}")
print(f"Val   : {result['val']}   rows -> {result['paths']['val']}")
print(f"Test  : {result['test']}  rows -> {result['paths']['test']}")

Train : 5572 rows -> /Users/kaizu/All/Projects/BengaliLLM/data/processed/train.jsonl
Val   : 1195   rows -> /Users/kaizu/All/Projects/BengaliLLM/data/processed/val.jsonl
Test  : 1195  rows -> /Users/kaizu/All/Projects/BengaliLLM/data/processed/test.jsonl


## Cell 7 — Sanity check: read back one row from each split

In [9]:
import json

# Confirm exported files are readable and have correct schema
for split in ['train', 'val', 'test']:
    path = result['paths'][split]
    with open(path, encoding='utf-8') as f:
        row = json.loads(f.readline())
    print(f'--- {split.upper()} ---')
    print('Keys   :', list(row.keys()))
    print('id     :', row['id'])
    print('answer :', row['answer'])
    print('choices:', row['choices'])
    print()

--- TRAIN ---
Keys   : ['id', 'domain', 'question', 'choices', 'answer', 'source']
id     : edu_5364
answer : A
choices: {'A': '0-96708', 'B': '0-58932', 'C': '0-75292', 'D': '0-32195'}

--- VAL ---
Keys   : ['id', 'domain', 'question', 'choices', 'answer', 'source']
id     : edu_2844
answer : D
choices: {'A': '3.51', 'B': '4.37', 'C': '3.74', 'D': '2.32'}

--- TEST ---
Keys   : ['id', 'domain', 'question', 'choices', 'answer', 'source']
id     : edu_1927
answer : D
choices: {'A': '0, 0', 'B': '2i, $\\frac{\\pi}{2}$', 'C': '-2i, -$\\pi$', 'D': '-2, -$\\pi$'}

